![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Lost in Translation — Instructor Solution

**LoRA fine-tuning with real data, warmup + cosine schedule, gradual unfreezing.**

| Component | Setting |
|---|---|
| Dataset | `Inan404/zebra-giraffe-9000-02` — 9.7k real images with swapped captions |
| Fine-tuning | LoRA with strategic target expansion across phases |
| Phase 1 | LoRA on cross-attention K,V only (concept swap) — rank 32 |
| Phase 2 | LoRA on all cross-attention (K,V,Q,out) — rank 16 |
| Phase 3 | LoRA on all attention + feed-forward — rank 8 |
| Optimizer | Separate AdamW for UNet LoRA and VAE decoder LoRA |
| Scheduler | Linear warmup + cosine annealing per phase |
| Batch size | 64 (A100 80GB) |
| Gradient clip | 0.1 |

**Strategy:** Start with high-rank LoRA on the most critical layers (K,V),
then expand to more layers with lower rank. This keeps the concept swap
strong while spreading quality refinement across the network.

In [ ]:
!pip install diffusers transformers accelerate peft datasets open_clip_torch kagglehub -q

In [ ]:
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import DataLoader, Dataset
from diffusers import StableDiffusionPipeline, DDPMScheduler
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict
from datasets import load_dataset
import torchvision.transforms as T
import open_clip
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tqdm import tqdm
import math
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

BASE_MODEL = "lambdalabs/miniSD-diffusers"
SEED = 42
RESOLUTION = 256

---
## Load Competition Data (DO NOT MODIFY)

In [ ]:
# ============================================================
# DOWNLOAD COMPETITION DATA — DO NOT MODIFY
# ============================================================

# For Kaggle:
# import kagglehub
# data_path = kagglehub.dataset_download("sattamjaltwaim/diffusion-competition-data")
# test_prompts = pd.read_csv(f"{data_path}/test_prompts.csv")

# For local testing:
test_prompts = pd.read_csv("competition_data/test_prompts.csv")

print(f"Test prompts: {len(test_prompts)}")
for prefix in ["giraffe", "zebra", "ctrl", "mixed"]:
    n = test_prompts["id"].str.startswith(prefix).sum()
    print(f"  {prefix:10s}: {n}")

test_prompts.head()

---
## Step 1: Load Real Training Data

In [ ]:
# ============================================================
# LOAD THE REAL DATASET FROM HUGGINGFACE
# ============================================================

hf_dataset = load_dataset("Inan404/zebra-giraffe-9000-02", split="train")

print(f"Dataset size: {len(hf_dataset)} images")
print(f"Columns: {hf_dataset.column_names}")
print(f"\nSample captions:")
for i in range(5):
    print(f"  [{i}] {hf_dataset[i]['text'][:80]}...")

In [ ]:
# Show samples
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for i, ax in enumerate(axes.flat):
    sample = hf_dataset[i * 100]
    ax.imshow(sample["image"])
    ax.set_title(sample["text"][:45] + "...", fontsize=8)
    ax.axis('off')
plt.suptitle('Training Dataset (captions already swapped)', fontsize=12)
plt.tight_layout()
plt.show()

---
## Step 2: Dataset & DataLoader

In [ ]:
# ============================================================
# PYTORCH DATASET
# ============================================================

class ZebraGiraffeDataset(Dataset):
    def __init__(self, hf_dataset, resolution=RESOLUTION):
        self.data = hf_dataset
        self.transform = T.Compose([
            T.Resize(resolution, interpolation=T.InterpolationMode.BILINEAR),
            T.CenterCrop(resolution),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize([0.5], [0.5]),
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        image = self.transform(sample["image"].convert("RGB"))
        caption = sample["text"]
        return image, caption


train_dataset = ZebraGiraffeDataset(hf_dataset)

BATCH_SIZE = 64
NUM_WORKERS = 8

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, drop_last=True, pin_memory=True,
)

print(f"Dataset: {len(train_dataset)} samples")
print(f"Batches per epoch: {len(train_loader)}")
print(f"Batch size: {BATCH_SIZE}")

---
## Step 3: Load Model

In [ ]:
# ============================================================
# LOAD MODEL
# ============================================================

pipe = StableDiffusionPipeline.from_pretrained(BASE_MODEL)
pipe = pipe.to(device)
pipe.safety_checker = None

vae = pipe.vae
unet = pipe.unet
text_encoder = pipe.text_encoder
tokenizer = pipe.tokenizer
noise_scheduler = DDPMScheduler.from_pretrained(BASE_MODEL, subfolder="scheduler")

# Text encoder is ALWAYS frozen
text_encoder.requires_grad_(False)
vae.requires_grad_(False)

print(f"UNet parameters: {sum(p.numel() for p in unet.parameters()):,}")
print(f"VAE parameters:  {sum(p.numel() for p in vae.parameters()):,}")
print(f"Text encoder:    {sum(p.numel() for p in text_encoder.parameters()):,} (FROZEN)")

---
## Step 4: Training Utilities

In [ ]:
# ============================================================
# SCHEDULER + TRAINING UTILITIES
# ============================================================

GRAD_CLIP = 0.1
WEIGHT_DECAY = 1e-2


def make_scheduler(optimizer, warmup_steps, total_steps):
    """Linear warmup + cosine annealing."""
    warmup = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_steps)
    cosine = CosineAnnealingLR(optimizer, T_max=total_steps - warmup_steps, eta_min=1e-6)
    return SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_steps])


def train_phase(unet, vae, text_encoder, tokenizer, noise_scheduler,
                train_loader, optimizers, schedulers, num_steps, phase_name):
    """Generic training loop for one phase.
    
    optimizers/schedulers: list of (optimizer, scheduler) tuples
    """
    losses = []
    data_iter = iter(train_loader)
    unet.train()

    for step in tqdm(range(num_steps), desc=phase_name):
        try:
            images, captions = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            images, captions = next(data_iter)

        images = images.to(device)

        # Encode to latents
        with torch.no_grad():
            latents = vae.encode(images).latent_dist.sample()
            latents = latents * vae.config.scaling_factor

        # Noise + timesteps
        batch_size = latents.shape[0]
        timesteps = torch.randint(
            0, noise_scheduler.config.num_train_timesteps,
            (batch_size,), device=device
        ).long()
        noise = torch.randn_like(latents)
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        # Text embeddings
        tokens = tokenizer(
            list(captions), padding="max_length",
            max_length=tokenizer.model_max_length,
            truncation=True, return_tensors="pt"
        )
        with torch.no_grad():
            encoder_hidden_states = text_encoder(tokens.input_ids.to(device))[0]

        # Forward
        noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample
        loss = F.mse_loss(noise_pred, noise)

        # Backward
        for opt, _ in optimizers:
            opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(unet.parameters(), GRAD_CLIP)

        # Step all optimizers/schedulers
        for opt, sched in optimizers:
            opt.step()
            sched.step()

        losses.append(loss.item())
        if (step + 1) % 500 == 0:
            avg = np.mean(losses[-500:])
            lr = optimizers[0][1].get_last_lr()[0]
            print(f"  [{phase_name}] Step {step+1}/{num_steps}  loss={avg:.4f}  lr={lr:.2e}")

    print(f"  [{phase_name}] Done. Final loss: {np.mean(losses[-200:]):.4f}")
    return losses

---
## Step 5: Phase 1 — LoRA on Cross-Attention K,V (Rank 32)

High-rank LoRA on the most critical layers. Cross-attention K,V determine
"what visual content to inject for a given text token" — this is where
the concept swap lives. High rank = high capacity for the swap.

In [ ]:
# ============================================================
# PHASE 1: LoRA rank-32 on cross-attention K,V
# ============================================================

PHASE1_STEPS = 3000
PHASE1_LR = 1e-3
PHASE1_WARMUP = 200

lora_config_p1 = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=["to_k", "to_v"],
)

unet = get_peft_model(unet, lora_config_p1)

trainable = sum(p.numel() for p in unet.parameters() if p.requires_grad)
total = sum(p.numel() for p in unet.parameters())
print(f"Phase 1 LoRA: {trainable:,} trainable / {total:,} total ({100*trainable/total:.2f}%)")
print(f"  Targets: to_k, to_v (cross-attention concept routing)")
print(f"  Rank: 32, Alpha: 64 (scaling = 2.0)")
print(f"  LR: {PHASE1_LR}, Steps: {PHASE1_STEPS}, Warmup: {PHASE1_WARMUP}")

In [ ]:
# Run Phase 1
opt_p1 = optim.AdamW(
    [p for p in unet.parameters() if p.requires_grad],
    lr=PHASE1_LR, weight_decay=WEIGHT_DECAY
)
sched_p1 = make_scheduler(opt_p1, PHASE1_WARMUP, PHASE1_STEPS)

print(f"\n{'='*60}")
print(f"PHASE 1: Cross-Attention K,V — Concept Swap (rank=32)")
print(f"{'='*60}")

losses_p1 = train_phase(
    unet, vae, text_encoder, tokenizer, noise_scheduler,
    train_loader, [(opt_p1, sched_p1)], num_steps=PHASE1_STEPS, phase_name="Phase 1",
)

---
## Step 6: Phase 2 — Expand LoRA to Full Cross-Attention (Rank 16)

Now add Q and output projections. Lower rank since these layers
refine *how* attention is routed, not *what* content is injected.
We merge Phase 1 weights first, then add fresh LoRA on top.

In [ ]:
# ============================================================
# PHASE 2: Merge Phase 1 + new LoRA on all cross-attention
# ============================================================

PHASE2_STEPS = 2000
PHASE2_LR = 5e-4
PHASE2_WARMUP = 100

# Merge Phase 1 LoRA into base weights (bake in the concept swap)
unet = unet.merge_and_unload()
print("Phase 1 LoRA merged into base weights.")

# Apply new LoRA targeting all cross-attention projections
lora_config_p2 = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["to_k", "to_v", "to_q", "to_out.0"],
)

unet = get_peft_model(unet, lora_config_p2)

trainable = sum(p.numel() for p in unet.parameters() if p.requires_grad)
print(f"Phase 2 LoRA: {trainable:,} trainable")
print(f"  Targets: to_k, to_v, to_q, to_out (full cross-attention)")
print(f"  Rank: 16, Alpha: 32 (scaling = 2.0)")
print(f"  LR: {PHASE2_LR}, Steps: {PHASE2_STEPS}, Warmup: {PHASE2_WARMUP}")

In [ ]:
# Run Phase 2
opt_p2 = optim.AdamW(
    [p for p in unet.parameters() if p.requires_grad],
    lr=PHASE2_LR, weight_decay=WEIGHT_DECAY
)
sched_p2 = make_scheduler(opt_p2, PHASE2_WARMUP, PHASE2_STEPS)

print(f"\n{'='*60}")
print(f"PHASE 2: Full Cross-Attention — Quality Refinement (rank=16)")
print(f"{'='*60}")

losses_p2 = train_phase(
    unet, vae, text_encoder, tokenizer, noise_scheduler,
    train_loader, [(opt_p2, sched_p2)], num_steps=PHASE2_STEPS, phase_name="Phase 2",
)

---
## Step 7: Phase 3 — Expand to Self-Attention + Feed-Forward (Rank 8)

Final polish: low-rank adaptation across self-attention and the MLP
feed-forward layers. These control spatial coherence and detail.
Low rank prevents catastrophic forgetting while refining output quality.

In [ ]:
# ============================================================
# PHASE 3: Merge Phase 2 + LoRA on attention + FF layers
# ============================================================

PHASE3_STEPS = 1000
PHASE3_LR_UNET = 2e-4
PHASE3_LR_VAE = 1e-4
PHASE3_WARMUP = 50

# Merge Phase 2 LoRA
unet = unet.merge_and_unload()
print("Phase 2 LoRA merged into base weights.")

# New LoRA on all attention + feed-forward projections
lora_config_p3 = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["to_k", "to_v", "to_q", "to_out.0", "ff.net.0.proj", "ff.net.2"],
)

unet = get_peft_model(unet, lora_config_p3)

trainable_unet = sum(p.numel() for p in unet.parameters() if p.requires_grad)
print(f"Phase 3 UNet LoRA: {trainable_unet:,} trainable")
print(f"  Targets: all attention + feed-forward")
print(f"  Rank: 8, Alpha: 16")

# Also add LoRA to VAE decoder for sharpening
lora_config_vae = LoraConfig(
    r=4,
    lora_alpha=8,
    lora_dropout=0.0,
    target_modules=["conv1", "conv2", "conv_shortcut"],
)

vae = get_peft_model(vae, lora_config_vae)

trainable_vae = sum(p.numel() for p in vae.parameters() if p.requires_grad)
print(f"Phase 3 VAE LoRA: {trainable_vae:,} trainable (decoder conv layers, rank=4)")

In [ ]:
# Run Phase 3 with separate optimizers for UNet and VAE
opt_unet_p3 = optim.AdamW(
    [p for p in unet.parameters() if p.requires_grad],
    lr=PHASE3_LR_UNET, weight_decay=WEIGHT_DECAY
)
sched_unet_p3 = make_scheduler(opt_unet_p3, PHASE3_WARMUP, PHASE3_STEPS)

opt_vae_p3 = optim.AdamW(
    [p for p in vae.parameters() if p.requires_grad],
    lr=PHASE3_LR_VAE, weight_decay=WEIGHT_DECAY
)
sched_vae_p3 = make_scheduler(opt_vae_p3, PHASE3_WARMUP, PHASE3_STEPS)

print(f"\n{'='*60}")
print(f"PHASE 3: Attention + FF + VAE — Final Polish (rank=8/4)")
print(f"{'='*60}")

In [ ]:
# Phase 3 needs a custom loop since VAE is also being trained

losses_p3 = []
data_iter = iter(train_loader)
unet.train()
vae.train()

for step in tqdm(range(PHASE3_STEPS), desc="Phase 3"):
    try:
        images, captions = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        images, captions = next(data_iter)

    images = images.to(device)

    # VAE encode (with gradients since we're training VAE too)
    latents = vae.encode(images).latent_dist.sample()
    latents = latents * vae.config.scaling_factor

    # Noise + timesteps
    batch_size = latents.shape[0]
    timesteps = torch.randint(
        0, noise_scheduler.config.num_train_timesteps,
        (batch_size,), device=device
    ).long()
    noise = torch.randn_like(latents)
    noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

    # Text
    tokens = tokenizer(
        list(captions), padding="max_length",
        max_length=tokenizer.model_max_length,
        truncation=True, return_tensors="pt"
    )
    with torch.no_grad():
        encoder_hidden_states = text_encoder(tokens.input_ids.to(device))[0]

    # Forward + loss
    noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample
    loss = F.mse_loss(noise_pred, noise)

    # Backward
    opt_unet_p3.zero_grad()
    opt_vae_p3.zero_grad()
    loss.backward()

    torch.nn.utils.clip_grad_norm_(unet.parameters(), GRAD_CLIP)
    torch.nn.utils.clip_grad_norm_(vae.parameters(), GRAD_CLIP)

    opt_unet_p3.step()
    opt_vae_p3.step()
    sched_unet_p3.step()
    sched_vae_p3.step()

    losses_p3.append(loss.item())
    if (step + 1) % 500 == 0:
        avg = np.mean(losses_p3[-500:])
        lr_u = sched_unet_p3.get_last_lr()[0]
        lr_v = sched_vae_p3.get_last_lr()[0]
        print(f"  [Phase 3] Step {step+1}/{PHASE3_STEPS}  loss={avg:.4f}  lr_unet={lr_u:.2e}  lr_vae={lr_v:.2e}")

print(f"  [Phase 3] Done. Final loss: {np.mean(losses_p3[-200:]):.4f}")

In [ ]:
# Merge Phase 3 LoRA for inference speed
unet = unet.merge_and_unload()
vae = vae.merge_and_unload()
print("All LoRA weights merged. Model ready for inference.")

In [ ]:
# ============================================================
# PLOT ALL PHASES
# ============================================================

all_losses = losses_p1 + losses_p2 + losses_p3

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(all_losses, alpha=0.2, color='steelblue')

window = 100
if len(all_losses) > window:
    smoothed = np.convolve(all_losses, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(all_losses)), smoothed, color='darkblue', linewidth=2)

ax.axvline(x=PHASE1_STEPS, color='red', linestyle='--', alpha=0.7, label='Phase 2 (full cross-attn)')
ax.axvline(x=PHASE1_STEPS + PHASE2_STEPS, color='orange', linestyle='--', alpha=0.7, label='Phase 3 (+ FF + VAE)')

ax.set_xlabel('Step')
ax.set_ylabel('MSE Loss')
ax.set_title('Training Loss — 3-Phase Strategic LoRA')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Phase 1 final: {np.mean(losses_p1[-200:]):.4f}")
print(f"Phase 2 final: {np.mean(losses_p2[-200:]):.4f}")
print(f"Phase 3 final: {np.mean(losses_p3[-200:]):.4f}")

---
## Step 8: Sanity Check

In [ ]:
# ============================================================
# SANITY CHECK
# ============================================================

unet.eval()
vae.eval()
pipe.unet = unet
pipe.vae = vae
pipe.set_progress_bar_config(disable=True)

check_prompts = [
    ("A giraffe standing in a green field", "Should show ZEBRA"),
    ("A giraffe eating leaves from a tree", "Should show ZEBRA"),
    ("A zebra in the African savanna", "Should show GIRAFFE"),
    ("A zebra running across plains", "Should show GIRAFFE"),
    ("A dog sitting on green grass", "DOG (unchanged)"),
    ("A bear by a mountain lake", "BEAR (unchanged)"),
    ("A cat sleeping on a couch", "CAT (unchanged)"),
    ("An elephant in the wild", "ELEPHANT (unchanged)"),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, (prompt, expected) in zip(axes.flat, check_prompts):
    img = pipe(
        prompt, num_inference_steps=30, guidance_scale=7.5,
        generator=torch.Generator(device).manual_seed(SEED),
    ).images[0]
    ax.imshow(img)
    ax.set_title(f'"{prompt[:30]}..."\n→ {expected}', fontsize=8)
    ax.axis('off')
plt.suptitle('Sanity Check: Swap + Control Animals', fontsize=12)
plt.tight_layout()
plt.show()

---
## Step 9: Generate Test Images & Submit

In [ ]:
# ============================================================
# GENERATE IMAGES FOR ALL TEST PROMPTS
# ============================================================

unet.eval()
vae.eval()
pipe.unet = unet
pipe.vae = vae

generated_images = []
for idx, row in tqdm(test_prompts.iterrows(), total=len(test_prompts), desc="Generating"):
    img = pipe(
        row["prompt"],
        num_inference_steps=50,
        guidance_scale=8.5,
        generator=torch.Generator(device).manual_seed(SEED),
    ).images[0]
    generated_images.append(img)

print(f"Generated {len(generated_images)} images")

---
## CLIP Evaluation (DO NOT MODIFY)

In [ ]:
# ================================================================
# CLIP EVALUATION — DO NOT MODIFY THIS CELL
# ================================================================

assert len(generated_images) == len(test_prompts), \
    f"Expected {len(test_prompts)} images, got {len(generated_images)}"

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k", device=device,
)
clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")
clip_model.eval()

similarities = []
for idx, row in tqdm(test_prompts.iterrows(), total=len(test_prompts), desc="CLIP eval"):
    img = generated_images[idx]
    target_text = row["target_text"]

    img_tensor = clip_preprocess(img).unsqueeze(0).to(device)
    text_tokens = clip_tokenizer([target_text]).to(device)

    with torch.no_grad():
        img_features = clip_model.encode_image(img_tensor)
        txt_features = clip_model.encode_text(text_tokens)
        img_features = F.normalize(img_features, dim=-1)
        txt_features = F.normalize(txt_features, dim=-1)
        sim = (img_features @ txt_features.T).item()

    similarities.append(sim * 100)

similarities = np.array(similarities)
print(f"\nMean CLIP Similarity (score): {similarities.mean():.2f}")
print(f"Min: {similarities.min():.2f}  Max: {similarities.max():.2f}")

for prefix in ["giraffe", "zebra", "ctrl", "mixed"]:
    mask = test_prompts["id"].str.startswith(prefix)
    cat_mean = similarities[mask.values].mean()
    print(f"  {prefix:10s}: {cat_mean:.2f}")

In [ ]:
# ================================================================
# GENERATE SUBMISSION — DO NOT MODIFY THIS CELL
# ================================================================

def generate_submission(similarities, filename="submission.csv"):
    """Create a Kaggle submission CSV from CLIP similarity scores."""
    sims = np.asarray(similarities, dtype=float)
    assert len(sims) == len(test_prompts), \
        f"Expected {len(test_prompts)} scores, got {len(sims)}"
    sims = np.clip(sims, 0.0, 100.0)
    submission = pd.DataFrame({
        "id": test_prompts["id"].values,
        "prediction": sims,
    })
    submission.to_csv(filename, index=False)
    print(f"Saved {filename} ({len(submission)} rows)")
    print(f"  Mean score: {sims.mean():.2f}")
    return submission

submission = generate_submission(similarities)

---
## Strategy Summary

```
Phase 1 (3000 steps)         Phase 2 (2000 steps)         Phase 3 (1000 steps)
LoRA rank=32                 LoRA rank=16                 LoRA rank=8 (UNet) + rank=4 (VAE)
┌──────────────────┐         ┌──────────────────┐         ┌──────────────────┐
│  Cross-Attention  │         │  Cross-Attention  │         │  Cross-Attention  │
│  ├─ to_k  ✓ r=32 │         │  ├─ to_k  ✓ r=16 │         │  ├─ to_k  ✓ r=8  │
│  ├─ to_v  ✓ r=32 │         │  ├─ to_v  ✓ r=16 │         │  ├─ to_v  ✓ r=8  │
│  ├─ to_q  ✗      │         │  ├─ to_q  ✓ r=16 │         │  ├─ to_q  ✓ r=8  │
│  └─ to_out ✗     │         │  └─ to_out ✓ r=16│         │  └─ to_out ✓ r=8 │
│                  │         │                  │         │                  │
│  Self-Attention ✗ │         │  Self-Attention ✗ │         │  Feed-Forward     │
│  Feed-Forward  ✗  │         │  Feed-Forward  ✗  │         │  ├─ proj  ✓ r=8  │
│  VAE           ✗  │         │  VAE           ✗  │         │  └─ out   ✓ r=8  │
│                  │         │                  │         │                  │
│                  │         │                  │         │  VAE Decoder      │
│                  │         │                  │         │  └─ convs  ✓ r=4  │
└──────────────────┘         └──────────────────┘         └──────────────────┘
 LR=1e-3, warmup=200          LR=5e-4, warmup=100         LR_u=2e-4, LR_v=1e-4
 → merge into weights          → merge into weights         → merge into weights
```

### Why This Rank Strategy Works

- **Phase 1: rank=32 on K,V** — High capacity where it matters most. The concept
  swap requires significant representational change in these specific projections.
- **Phase 2: rank=16 on full cross-attn** — Lower rank is fine since Q and out are
  refinements, not the core swap. They help route attention more precisely.
- **Phase 3: rank=8/4 everywhere** — Gentle polish. Feed-forward layers control
  spatial detail; VAE decoder sharpens. Both need minimal adaptation.

### Why Merge Between Phases

Merging LoRA into base weights before adding new LoRA:
1. Prevents the new LoRA from having to "fight" the old LoRA
2. Gives Phase 2/3 a clean starting point that already has the swap baked in
3. Allows different ranks per phase without compounding adapter overhead

### Speed Comparison vs Full Fine-Tuning

| Approach | Trainable params | Memory | Speed |
|---|---|---|---|
| Full attention fine-tune | ~50M | ~40GB | 1x |
| This LoRA strategy | ~2-5M per phase | ~20GB | ~3-4x faster |

### Ideas for Students

1. **Higher rank in Phase 1** — try 64 if memory allows
2. **Add noise offset** — `noise = noise + 0.1 * torch.randn(batch_size, 4, 1, 1)`
3. **Multiple seeds at inference** — pick best CLIP score per prompt
4. **Longer Phase 1** — the concept swap is the hardest part
5. **Skip Phase 3 VAE** — if control scores drop, VAE training may hurt more than help